# 08 — Enriched Chunks and Search Preparation

## Objective

Notebook 07 produced chunks -- fragments of a document's text with no memory of where they came from. This notebook adds that memory back, and makes the point that "adding context back to a chunk" is not one single operation: what gets embedded for search and what gets shown to the LLM after retrieval can, and often should, be two different strings, each built from the same underlying chunk for a different purpose.

## What We Will Learn

- Why a bare chunk (`chunk_text`, as produced in Notebook 07) is frequently missing exactly the context a reader or a search index needs
- How to build an **enriched chunk** by attaching document-level metadata back onto each chunk
- Why `chunk_to_embed` (what gets embedded and indexed) and `chunk_to_retrieve` (what gets shown to the LLM after a match) are deliberately different pieces of text, not two names for the same string
- The "small-to-big" retrieval pattern: embedding a narrow, precise unit of text while retrieving a wider one built around it
- What `ai_prep_search` is meant to do conceptually, and why this notebook treats its exact signature as unverified

## Prerequisites

- Completed `07_document_chunking_basics.ipynb`, **re-run with its `chunk_size` widget set lower than the default `400`** -- for example `150`. At `chunk_size=400`, every one of Notebook 02's 15 documents fits in a single chunk (each document's `content` is only 280-380 characters), which leaves nothing for Step 6 below to stitch together. At `chunk_size=150`, every document splits into 3-4 chunks. This was checked by running `recursive_chunk` against all 15 documents' actual text before writing this notebook, not assumed.
- This notebook reads the `document_chunks` table Notebook 07 writes, and writes a new `enriched_chunks` table alongside it
- Basic Python and PySpark, same level as Notebook 07. No Databricks AI Functions are required for Steps 1-8 -- only Step 9 touches one, and its exact behavior is explicitly unverified (see that step and Common Errors / Limitations)

## Conceptual Explanation

**Why enrichment.** Chunking (Notebook 07) decides where a document gets cut. It says nothing about what a reader -- human, embedding model, or LLM -- can still tell about a chunk once it's separated from the rest of its document. A chunk like "Flagged transactions are reviewed by an AML analyst within two business days" reads fine in isolation. A chunk that is the back half of a sentence, or that leans on a pronoun resolved several sentences earlier, does not. Enrichment is the general term for putting some of that lost context back -- typically the source document's own metadata -- onto each chunk, without changing how the chunking itself was done.

**Enriched chunk content.** The simplest form of enrichment: prepend a short metadata header (title, category, department, and this chunk's position among its document's chunks) to `chunk_text`. The same header shape Notebook 06 used at the whole-document level, applied here per chunk. Step 4 builds this directly.

**`chunk_to_embed` versus `chunk_to_retrieve` -- the key idea.** Nothing requires the text an embedding gets computed from to be the exact text shown to the LLM once that embedding's chunk is retrieved. In fact, usually it shouldn't be:

- A vector index is trying to match a user's query against a chunk's *topic*. A short prefix of context (title + category) can help a generic-sounding sentence get matched to the right document. But padding the embedded text with a lot of extra material -- several neighboring chunks, say -- dilutes the resulting vector, and a diluted vector tends to match many things a little instead of one thing precisely.
- Once a chunk has been identified as the best match, there's no reason to show the LLM only that narrow slice. The chunk that scored highest was cut at an arbitrary character boundary back in Notebook 07 -- handing the LLM its immediate neighbors too means a sentence that happened to fall right at a chunk boundary is still there, in full, for the LLM to read.

This is sometimes called **small-to-big retrieval**: embed a small, precise unit of text; retrieve a larger, more complete one built around whichever small unit actually matched. Step 5 builds `chunk_to_embed` as a compact, context-tagged version of a single chunk; Step 6 builds `chunk_to_retrieve` by stitching a chunk together with its immediate `chunk_id` neighbors from the same document. In a real Vector Search index (Notebook 10), this means the embedded/indexed column is `chunk_to_embed`, while `chunk_to_retrieve` (or a reference that reconstructs it) is carried as an unindexed payload column returned alongside the match.

**`ai_prep_search`.** The name suggests a Databricks AI Function meant to normalize or prepare text specifically for search indexing -- the same naming family as `ai_parse_document`, `ai_classify`, and `ai_extract`. Unlike those three, though, this notebook has meaningfully less confidence in its exact current signature, GA status, or availability. Step 9 sketches an illustrative call, clearly marked as unverified rather than a reference to be relied on -- see that step and Common Errors / Limitations.

**Everything through Step 8 is plain Python and PySpark.** No AI Function and no embedding model is called until Step 9 -- enrichment, and the split between `chunk_to_embed` and `chunk_to_retrieve`, are string-building decisions that are logically independent of how the embedding itself later gets computed (Notebook 09).

## Example Data

The `document_chunks` table written by `07_document_chunking_basics.ipynb` (re-run with a smaller `chunk_size` -- see Prerequisites), joined back to the `documents` table from Notebook 02 to pull in `department`, which Notebook 07's chunks table doesn't itself carry. One document with several chunks is picked automatically as a running example carried through Steps 3-7; Step 8 then applies the same logic to every chunk of all 15 documents.

## Implementation

### Step 1 — Parameterize catalog and schema, point at Notebook 07's output

In [ ]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

documents_table = f"{catalog_name}.{schema_name}.documents"
chunks_table = f"{catalog_name}.{schema_name}.document_chunks"
enriched_table = f"{catalog_name}.{schema_name}.enriched_chunks"

print(f"Documents table:              {documents_table}")
print(f"Chunks table (Notebook 07):   {chunks_table}")
print(f"Enriched table (this Step 8): {enriched_table}")

### Step 2 — Load chunks, join back to `documents` for department, and pick a running example

`document_chunks` carries `title` and `category` but not `department` -- a realistic reminder that a chunks table won't always carry every field you eventually want, and a join back to the source table is sometimes simply required. The chunk-count check below is not decorative: if it reports every document has exactly one chunk, Step 6 has nothing to stitch together, and the fix is the one described in Prerequisites, not a code change here.

In [ ]:
document_chunks_df = spark.table(chunks_table)
documents_df = spark.table(documents_table)

chunks_with_department_df = document_chunks_df.join(
    documents_df.select("doc_id", "department"),
    on="doc_id",
    how="left",
).orderBy("doc_id", "chunk_id")
chunks_with_department_df.createOrReplaceTempView("chunks_v")

chunk_counts_df = chunks_with_department_df.groupBy("doc_id", "title").count().orderBy("doc_id")
display(chunk_counts_df)

max_chunks_per_doc = chunk_counts_df.agg({"count": "max"}).first()[0]
if max_chunks_per_doc < 2:
    print(
        "WARNING: every document has only 1 chunk. Re-run 07_document_chunking_basics.ipynb "
        "with its 'chunk_size' widget set lower (e.g. 150), then re-run this notebook from Step 1 -- "
        "Step 6 below has nothing to demonstrate if no document has neighboring chunks."
    )
else:
    print(f"OK: at least one document has {max_chunks_per_doc} chunks.")

# Pick the document with the most chunks as the running example for Steps 3-7
example_doc_id = chunk_counts_df.orderBy(chunk_counts_df["count"].desc()).first()["doc_id"]
example_chunks = chunks_with_department_df.filter(f"doc_id = {example_doc_id}").orderBy("chunk_id").collect()

print(f"\nUsing doc_id={example_doc_id} ({example_chunks[0]['title']}) as the running example: {len(example_chunks)} chunks")

### Step 3 — Original chunk content

`chunk_text`, exactly as Notebook 07 produced it -- no title, no category, no signal at all about which document or section it came from. Read each one and judge for yourself which would be interpretable if it showed up alone in a search result, with nothing else attached.

The middle chunk (`target_idx`) is selected here and reused through Steps 4-7, so the same piece of text can be compared across every transformation.

In [ ]:
for row in example_chunks:
    print(f"\n--- chunk {row['chunk_id']} ({len(row['chunk_text'])} chars) ---")
    print(row["chunk_text"])

# The middle chunk has both a previous and a next neighbor within its document -- the most
# informative case for Step 6's neighbor-stitching. Falls back to the first chunk if there are
# fewer than 3.
target_idx = 1 if len(example_chunks) >= 3 else 0
target_row = example_chunks[target_idx]
print(f"\nTarget chunk for Steps 4-7: chunk_id={target_row['chunk_id']} (index {target_idx} of {len(example_chunks)})")

### Step 4 — Enriched chunk content

A four-line metadata header -- title, category, department, and this chunk's position among its document's chunks -- prepended to `chunk_text`.

In [ ]:
def build_enriched_chunk(row, chunk_position, total_chunks):
    return (
        f"Title: {row['title']}\n"
        f"Category: {row['category']}\n"
        f"Department: {row['department']}\n"
        f"Chunk: {chunk_position + 1} of {total_chunks}\n\n"
        f"{row['chunk_text']}"
    )


enriched_text = build_enriched_chunk(target_row, target_idx, len(example_chunks))
print(f"chunk_enriched ({len(enriched_text)} chars), vs chunk_text ({len(target_row['chunk_text'])} chars):\n")
print(enriched_text)

### Step 5 — Content used for embeddings: `chunk_to_embed`

Deliberately *not* the same as the enriched chunk above. Where Step 4's header is a full four-line block meant for a human or an LLM to read, `chunk_to_embed` uses a short single-line prefix -- just enough context to help a generic sentence get matched to the right document, without burying the embedding in repeated metadata every one of this document's chunks would otherwise share.

In [ ]:
def build_chunk_to_embed(row):
    return f"[{row['title']} | {row['category']}] {row['chunk_text']}"


chunk_to_embed_text = build_chunk_to_embed(target_row)
print(f"chunk_to_embed ({len(chunk_to_embed_text)} chars), vs chunk_enriched ({len(enriched_text)} chars):\n")
print(chunk_to_embed_text)

### Step 6 — Content returned to the LLM: `chunk_to_retrieve`

Stitches the target chunk together with its immediate `chunk_id` neighbors from the same document (one before, one after, when they exist), plus a citation-style header. This is deliberately *wider* than `chunk_to_embed` -- the small-to-big pattern from the Conceptual Explanation. Neighbor lookup never crosses a `doc_id` boundary, so a chunk at the very start or end of its document only gets one neighbor instead of two.

In [ ]:
def build_chunk_to_retrieve(doc_chunks, idx):
    row = doc_chunks[idx]
    neighbor_idxs = [i for i in (idx - 1, idx, idx + 1) if 0 <= i < len(doc_chunks)]
    combined_text = " ".join(doc_chunks[i]["chunk_text"] for i in neighbor_idxs)
    header = f"Source: {row['title']} ({row['category']}, {row['department']})\n\n"
    return header + combined_text, [doc_chunks[i]["chunk_id"] for i in neighbor_idxs]


chunk_to_retrieve_text, stitched_chunk_ids = build_chunk_to_retrieve(example_chunks, target_idx)
print(f"chunk_to_retrieve ({len(chunk_to_retrieve_text)} chars), built from chunk_ids {stitched_chunk_ids}:\n")
print(chunk_to_retrieve_text)

### Step 7 — Compare all four variants side by side

Same source chunk, four derived texts, four different sizes. `avg`/`min`/`max` aren't needed here -- with one chunk there are only four numbers to compare directly.

In [ ]:
comparison_rows = [
    ("chunk_text (original)", len(target_row["chunk_text"])),
    ("chunk_enriched", len(enriched_text)),
    ("chunk_to_embed", len(chunk_to_embed_text)),
    ("chunk_to_retrieve", len(chunk_to_retrieve_text)),
]
comparison_df = spark.createDataFrame(comparison_rows, ["variant", "char_count"])
display(comparison_df)

print(f"\nSide by side, for chunk_id={target_row['chunk_id']} of doc_id={example_doc_id}:\n")
for label, text in [
    ("ORIGINAL", target_row["chunk_text"]),
    ("ENRICHED", enriched_text),
    ("TO EMBED", chunk_to_embed_text),
    ("TO RETRIEVE", chunk_to_retrieve_text),
]:
    print(f"--- {label} ---")
    print(text)
    print()

### Step 8 — Build and persist `enriched_chunks` for the full corpus

The same four functions from Steps 4-6, applied to every chunk of all 15 documents rather than just the running example. Neighbor stitching is computed per document (grouping `chunks_with_department_df`'s rows by `doc_id`, preserving `chunk_id` order) so `chunk_to_retrieve` never reaches across documents. The result is written to a new `enriched_chunks` table -- the reusable output this notebook produces for Notebook 09 onward, the same role Notebook 07's `document_chunks` played for this one.

In [ ]:
from collections import defaultdict
from pyspark.sql.functions import avg

all_rows = chunks_with_department_df.orderBy("doc_id", "chunk_id").collect()
rows_by_doc = defaultdict(list)
for r in all_rows:
    rows_by_doc[r["doc_id"]].append(r)

enriched_records = []
for doc_id, doc_chunks in rows_by_doc.items():
    total = len(doc_chunks)
    for i, row in enumerate(doc_chunks):
        row_enriched = build_enriched_chunk(row, i, total)
        row_to_embed = build_chunk_to_embed(row)
        row_to_retrieve, _ = build_chunk_to_retrieve(doc_chunks, i)
        enriched_records.append((
            row["doc_id"], row["chunk_id"], row["title"], row["category"], row["department"],
            row["chunk_text"], row_enriched, row_to_embed, row_to_retrieve,
            len(row["chunk_text"]), len(row_to_embed), len(row_to_retrieve),
        ))

enriched_columns = [
    "doc_id", "chunk_id", "title", "category", "department",
    "chunk_text", "chunk_enriched", "chunk_to_embed", "chunk_to_retrieve",
    "chunk_text_chars", "chunk_to_embed_chars", "chunk_to_retrieve_chars",
]
enriched_chunks_df = spark.createDataFrame(enriched_records, enriched_columns)

enriched_chunks_df.write.mode("overwrite").saveAsTable(enriched_table)
enriched_chunks_df.createOrReplaceTempView("enriched_chunks_v")
print(f"Wrote {enriched_chunks_df.count()} rows to {enriched_table}")

display(enriched_chunks_df.orderBy("doc_id", "chunk_id"))

display(
    enriched_chunks_df.select(
        avg("chunk_text_chars").alias("avg_chunk_text_chars"),
        avg("chunk_to_embed_chars").alias("avg_chunk_to_embed_chars"),
        avg("chunk_to_retrieve_chars").alias("avg_chunk_to_retrieve_chars"),
    )
)

### Step 9 — `ai_prep_search`, conceptually (unverified sketch)

Everything above builds `chunk_to_embed` and `chunk_to_retrieve` with plain Python string concatenation. Databricks' `ai_` function family suggests there may be a purpose-built function for preparing or normalizing text specifically for search indexing -- `ai_prep_search`. This notebook has **not** executed it against a live workspace, and unlike `ai_parse_document` / `ai_classify` / `ai_extract` in Notebooks 04-06, has meaningfully less confidence in its exact current name, signature, or availability. Treat the cell below as illustrative of the *shape* such a call might take, not a verified reference -- check your workspace's current documentation (or `DESCRIBE FUNCTION ai_prep_search`, if it exists there) before relying on it. If this cell fails with `UNRESOLVED_ROUTINE`, that is an expected possible outcome, not necessarily a configuration problem -- see Common Errors / Limitations.

In [ ]:
%sql
-- ILLUSTRATIVE ONLY: verify ai_prep_search's real name, signature, and availability in your
-- workspace before relying on this. It is not used anywhere else in this notebook, and its
-- output is not written to the enriched_chunks table above.
SELECT
  doc_id,
  chunk_id,
  chunk_to_embed,
  ai_prep_search(chunk_to_embed) AS chunk_to_embed_prepared
FROM enriched_chunks_v
LIMIT 5

## Inspect the Output

- Look back at Step 3's raw chunks for the example document. Which one(s), shown alone with no other context, would be hard to interpret? Which ones stand fine on their own?
- Compare Step 4's enriched chunk to the original. Does the added header make it easier to guess this chunk's source document, without reading `chunk_text` itself at all?
- Compare Step 5's `chunk_to_embed` to Step 4's enriched chunk. Which is shorter? Which do you expect would produce a sharper embedding for a narrow query, and why?
- In Step 6, which `chunk_id`s actually got stitched into `chunk_to_retrieve`? Did the target chunk have two neighbors, or only one because it was first or last in its document?
- In Step 7's comparison table, rank all four variants by `char_count`. Does the ranking match what the Conceptual Explanation predicted (`chunk_to_retrieve` largest, `chunk_text` smallest)?
- In Step 8's averages, how much bigger is `avg_chunk_to_retrieve_chars` than `avg_chunk_text_chars`? Does that roughly match what you'd expect, given most chunks pull in up to two neighbors out of 3-4 total?

## Experimentation Section

1. In Step 3, change `target_idx` to `0` (the first chunk) instead of the middle one, then re-run Steps 4-7. `chunk_to_retrieve` should now have only one neighbor instead of two -- confirm that in the printed `chunk_id` list.
2. Widen the neighbor window in `build_chunk_to_retrieve` from `(idx - 1, idx, idx + 1)` to two chunks on each side, and re-run Step 8. How much does `avg_chunk_to_retrieve_chars` grow?
3. Change `build_chunk_to_embed` to drop `category` and keep only `title`. Re-run Step 8 -- does that change how many chunks of the same document end up with near-identical `chunk_to_embed` prefixes?
4. Re-run `07_document_chunking_basics.ipynb` with `chunk_size=100` instead of `150`, then re-run this notebook end to end. Do more chunks now pull in the full two-neighbor window in `chunk_to_retrieve`?
5. If `ai_prep_search` is available in your workspace, run Step 9 for real, inspect exactly what it returns, and update this notebook's Conceptual Explanation and Common Errors sections with what you found -- what's written here about it is a documented guess, not a verified reference.

## Common Errors / Limitations

- **`chunk_to_embed` and `chunk_to_retrieve` here are one hand-designed choice, not the only correct answer** -- a short title+category prefix for embedding and a ±1-neighbor citation block for retrieval is a reasonable design, not a Databricks-prescribed standard. Real systems tune both based on measured retrieval quality (Notebook 17).
- **Neighbor stitching never crosses a `doc_id` boundary** -- intentional (pulling in an unrelated document's chunk would be wrong), but it means the first and last chunk of every document systematically get less surrounding context than middle chunks do.
- **Nothing here calls an embedding model** -- Steps 1-8 are plain Python/PySpark string building. Enrichment and the `chunk_to_embed`/`chunk_to_retrieve` split are decisions about *what text* to embed and retrieve, independent of *how* the embedding gets computed; that starts in Notebook 09.
- **`ai_prep_search`'s exact behavior is unverified** -- see Step 9. If its name, signature, or return shape differs from what's shown there, that is expected; this notebook was written without access to a live workspace to confirm it, and with less confidence in the starting assumption than Notebook 06 had for `ai_extract`.
- **`enriched_chunks` is overwritten on every run** -- `.mode("overwrite")` in Step 8 means re-running this notebook after changing `build_chunk_to_embed` or `build_chunk_to_retrieve` replaces the whole table rather than versioning it.
- **A wider `chunk_to_retrieve` isn't free** -- every extra neighbor stitched in is more text an LLM reads (and pays for) per retrieved chunk. Small-to-big retrieval trades some of fixed-size chunking's storage/compute cost (Notebook 07) for a similar cost at retrieval time instead.

## Summary

You took Notebook 07's bare chunks and built three deliberately different derived texts from each one: an enriched chunk (metadata header plus chunk), a compact `chunk_to_embed` meant to produce a precise, topic-tagged embedding, and a wider `chunk_to_retrieve` that stitches a chunk together with its immediate neighbors so the LLM sees more context than whatever narrow slice actually matched a query. The core idea to carry forward: what gets embedded and what gets shown to the LLM are two different design decisions, built from the same source chunk for two different jobs -- and both were persisted into a new `enriched_chunks` table alongside the original `chunk_text`. Step 9 sketched, without verifying, how a purpose-built `ai_prep_search` function might fit into this same pipeline.

## Suggested Exercises

- For one document in `enriched_chunks`, read `chunk_to_retrieve` for every one of its chunks back to back. Do the neighbor overlaps make the reconstructed text redundant, or does each chunk's retrieval window still add something the others didn't cover?
- Design a fourth text variant, `chunk_to_display`, meant for showing a chunk directly to a human user in a UI rather than to an LLM prompt. What would you include that `chunk_to_retrieve` doesn't need, or leave out that it does?
- When you're ready, move on to **`09_embeddings_basics.ipynb`**, which finally puts an actual embedding model behind `chunk_to_embed` and asks what a vector representation of this text looks like, and how "similar" two chunks' embeddings really are.